# Fase 3 — Relações ERA5 × Radar

Objetivo: medir associações descritivas entre os preditores ERA5 (originais e derivados) e o radar. O radar é analisado no domínio `expm1(target)`, chamado aqui de **unidades numéricas da legenda do radar** até confirmação documental da unidade física. Correlação e informação mútua não implicam causalidade.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

def resolve_output(rel):
    candidates = [Path(rel), Path("..") / rel, Path.cwd() / rel, Path.cwd().parent / rel]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return Path(rel).resolve()

OUT = resolve_output("analysis_outputs/03_joint")
print("Usando resultados em:", OUT)


In [ ]:
def load_parquet(name):
    path = OUT / name
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

def load_json(name):
    path = OUT / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

summary=load_json("analysis_summary.json")
catalog=load_parquet("predictor_catalog.parquet")
metrics=load_parquet("association_metrics.parquet")
bins=load_parquet("conditional_bins.parquet")
npz=OUT/"relationship_samples.npz"
data=np.load(npz,allow_pickle=False) if npz.exists() else None
if data is not None:
    X=data["predictors"]; radar=data["radar"]; names=[str(v) for v in data["predictor_names"].tolist()]


## 1. Resumo da amostra conjunta

In [ ]:
pd.DataFrame([summary])

## 2. Métricas de associação

In [ ]:
cols=["predictor","source","pearson_r","spearman_rho","mutual_information_regression","point_biserial_r_positive","mutual_information_positive_event","n_positive","spearman_rho_positive"]
metrics[cols].sort_values("mutual_information_regression",ascending=False) if not metrics.empty else metrics

### 2.1 Associação monotônica (Spearman)

In [ ]:
if not metrics.empty:
    d=metrics.copy(); d["abs_spearman"]=d["spearman_rho"].abs(); d=d.sort_values("abs_spearman")
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(d["predictor"],d["abs_spearman"]); ax.set_xlabel("|Spearman rho|"); plt.show()

### 2.2 Informação mútua com intensidade do radar

In [ ]:
if not metrics.empty:
    d=metrics.sort_values("mutual_information_regression")
    fig,ax=plt.subplots(figsize=(10,7)); ax.barh(d["predictor"],d["mutual_information_regression"]); ax.set_xlabel("Mutual information"); plt.show()

## 3. Probabilidade de evento por bins do preditor

In [ ]:
if not metrics.empty and not bins.empty:
    chosen=metrics.sort_values("mutual_information_positive_event",ascending=False)["predictor"].head(6).tolist()
    fig,axes=plt.subplots(3,2,figsize=(13,12)); axes=axes.ravel()
    for ax,name in zip(axes,chosen):
        d=bins[bins.predictor==name].sort_values("x_mean")
        ax.plot(d["x_mean"],d["positive_rate"],marker="o",label=">0")
        ax.plot(d["x_mean"],d["ge20_rate"],marker="o",label=">=20")
        ax.plot(d["x_mean"],d["ge30_rate"],marker="o",label=">=30")
        ax.set_title(name); ax.set_ylabel("Taxa de evento"); ax.legend()
    fig.tight_layout(); plt.show()

## 4. Relações bivariadas — hexbin

In [ ]:
if data is not None and not metrics.empty:
    chosen=metrics.sort_values("mutual_information_regression",ascending=False)["predictor"].head(6).tolist()
    fig,axes=plt.subplots(3,2,figsize=(13,12)); axes=axes.ravel()
    for ax,name in zip(axes,chosen):
        j=names.index(name)
        hb=ax.hexbin(X[:,j],radar,gridsize=60,mincnt=1,bins="log")
        ax.set_xlabel(name); ax.set_ylabel("Radar (unidades da legenda)"); ax.set_title(name)
        fig.colorbar(hb,ax=ax,label="log contagem")
    fig.tight_layout(); plt.show()

## 5. Relação somente durante radar positivo

In [ ]:
if not metrics.empty:
    cols=["predictor","n_positive","pearson_r_positive","spearman_rho_positive","mutual_information_regression_positive"]
    display(metrics[cols].sort_values("mutual_information_regression_positive",ascending=False))

## 6. Síntese da Fase 3

Use os resultados para selecionar variáveis que merecem investigação física detalhada na Fase 4. Relações fracas globalmente podem ainda ser importantes nos extremos, pois a massa de radar igual a zero domina a distribuição total.

In [ ]:
pd.DataFrame([
    {"checagem":"Amostra conjunta disponível","status":"OK" if data is not None else "REVISAR"},
    {"checagem":"20 preditores esperados","status":"OK" if len(catalog)>=20 else "REVISAR"},
    {"checagem":"Métricas calculadas","status":"OK" if len(metrics)>=20 else "REVISAR"},
    {"checagem":"Bins condicionais","status":"OK" if len(bins)>0 else "REVISAR"},
])